In [18]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [19]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [20]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [21]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [22]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [23]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [24]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [25]:
# dagshub setup

%pip install dagshub
import dagshub
dagshub.init(repo_owner='Rashi-nitor', repo_name='mlflow_dagshub-1', mlflow=True)

Note: you may need to restart the kernel to use updated packages.


Initialized MLflow to track repo "Rashi-nitor/mlflow_dagshub-1"

Repository Rashi-nitor/mlflow_dagshub-1 initialized!

In [26]:
# Run cells 4 and 5 first so models and reports are available.
mlflow.set_tracking_uri("https://dagshub.com/Rashi-nitor/mlflow_dagshub-1.mlflow")
mlflow.set_experiment("Anomaly Detection")  

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            # Trust only the tree type used by the locally trained RandomForest model.
            mlflow.sklearn.log_model(
                model,
                "model",
                skops_trusted_types=["sklearn.tree._tree.Tree"],
            ) 


2026/10/19 12:37:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: https://dagshub.com/Rashi-nitor/mlflow_dagshub-1.mlflow/#/experiments/0/runs/01aa9e630848427d99fea87d89b4315a
🧪 View experiment at: https://dagshub.com/Rashi-nitor/mlflow_dagshub-1.mlflow/#/experiments/0


2026/10/19 12:37:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: https://dagshub.com/Rashi-nitor/mlflow_dagshub-1.mlflow/#/experiments/0/runs/e410f42587744387a7d907252ff49992
🧪 View experiment at: https://dagshub.com/Rashi-nitor/mlflow_dagshub-1.mlflow/#/experiments/0


2026/10/19 12:38:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: https://dagshub.com/Rashi-nitor/mlflow_dagshub-1.mlflow/#/experiments/0/runs/46efe48911cc496a918414873a632904
🧪 View experiment at: https://dagshub.com/Rashi-nitor/mlflow_dagshub-1.mlflow/#/experiments/0


2026/10/19 12:38:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier With SMOTE at: https://dagshub.com/Rashi-nitor/mlflow_dagshub-1.mlflow/#/experiments/0/runs/7fd59e8e62544955a13a9c32c55491e2
🧪 View experiment at: https://dagshub.com/Rashi-nitor/mlflow_dagshub-1.mlflow/#/experiments/0
